# noisechisel-torch vs Gnuastro `astnoisechisel` — GPU speed & product comparison

Runs on **Google Colab** (Runtime → Change runtime type → **GPU**). The notebook:

1. installs `noisechisel-torch` from this repo and Gnuastro 0.24 (via micromamba, no kernel restart),
2. downloads three FITS frames from Google Drive,
3. runs Gnuastro (CPU), torch on CPU and torch on GPU, timing each,
4. compares every product (detections, Sky, Sky STD, input-no-sky, S/N threshold, STD stats) and plots the differences.

## 1. Setup

In [ ]:
import subprocess, torch
print(subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout or "NO GPU - switch the runtime to GPU")
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())

In [ ]:
%%capture
!pip install -q gdown astropy scipy
!pip install -q "noisechisel-torch[fits] @ git+https://github.com/PabloMSanAla/noisechisel.git"

Gnuastro 0.24 from conda-forge, installed in an isolated prefix with micromamba (the system `apt` Gnuastro is much older and would not be bit-comparable).

In [ ]:
%%bash
set -e
if [ ! -x /opt/gnu/bin/astnoisechisel ]; then
  mkdir -p /opt/mm && cd /opt/mm
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
  /opt/mm/bin/micromamba create -y -q -p /opt/gnu -c conda-forge gnuastro=0.24 libblas=*=*openblas
fi
/opt/gnu/bin/astnoisechisel --version | head -2

## 2. Download the frames

In [ ]:
import gdown, os
FILES = {
    "frame1": "1KVOIrK4gzOQQ_UwNTRicQDWzzD_Vvp-E",
    "frame2": "1EcCdtAWNHgXXBFj5ZigiKDgbSL1yFOGF",
    "frame3": "1Kqu8yRDW_PM_QkBwKFs-G1I1L2l95wib",
}
DATA = "/content/frames"; os.makedirs(DATA, exist_ok=True)
paths = {}
for name, fid in FILES.items():
    p = f"{DATA}/{name}.fits"
    if not os.path.exists(p):
        gdown.download(id=fid, output=p, quiet=True)
    paths[name] = p
    print(name, f"{os.path.getsize(p)/1e6:.1f} MB")

In [ ]:
from astropy.io import fits
import numpy as np

def find_hdu(path):
    with fits.open(path) as hl:
        for i, h in enumerate(hl):
            if h.data is not None and h.data.ndim == 2:
                return i
    raise ValueError(f"no 2D image in {path}")

hdus = {n: find_hdu(p) for n, p in paths.items()}
for n, p in paths.items():
    with fits.open(p) as hl:
        h = hl[hdus[n]]
        print(f"{n}: hdu={hdus[n]} shape={h.data.shape} dtype={h.data.dtype} NaNs={int(np.isnan(h.data).sum())}")

## 3. Run everything

* **Gnuastro**: `astnoisechisel` with default options and all CPU cores. Wall time of the subprocess (includes FITS read/write).
* **torch**: compute time only (image already in memory), best of `REPEAT` runs, with `ccl="scipy"` (connected components on the CPU; the default `ccl="torch"` on CUDA is very slow for large frames), with a warm-up run first (CUDA init / kernel compile is excluded). `torch.cuda.synchronize()` is called before stopping the clock.
* **torch end-to-end** (`run_fits`: read FITS → compute → write FITS) is also reported for a fairer comparison with Gnuastro's wall time.

In [ ]:
import time, shutil, subprocess, os, torch
from noisechisel_torch import NoiseChisel, read_image
from noisechisel_torch.io import run_fits

GNU = "/opt/gnu/bin/astnoisechisel"
OUT = "/content/out"; os.makedirs(OUT, exist_ok=True)
NTHREADS = os.cpu_count()
REPEAT = 2
# Connected-component labelling: "scipy" labels on the CPU (fast, identical labels) while everything else
# runs on the GPU.  "torch" (the CUDA default) uses iterative label propagation and can take many minutes.
CCL = "scipy"
devices = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])
print("CPU threads:", NTHREADS, "| ccl:", CCL, "| torch devices:", devices)

def sync(dev):
    if dev == "cuda": torch.cuda.synchronize()

def run_gnuastro(name):
    out = f"{OUT}/{name}_gnuastro.fits"
    if os.path.exists(out): os.remove(out)
    cmd = [GNU, paths[name], f"--hdu={hdus[name]}", f"--output={out}", "--quiet", f"--numthreads={NTHREADS}"]
    times = []
    for _ in range(REPEAT):
        if os.path.exists(out): os.remove(out)
        t0 = time.perf_counter(); subprocess.run(cmd, check=True); times.append(time.perf_counter() - t0)
    return out, min(times)

results, timing = {}, []
imgs = {n: read_image(p, hdus[n])[0] for n, p in paths.items()}

for name in paths:
    row = {"image": name, "shape": imgs[name].shape}
    gout, row["gnuastro_s"] = run_gnuastro(name)
    results[(name, "gnuastro")] = gout
    for dev in devices:
        nc = NoiseChisel(device=dev, ccl=CCL)
        t0 = time.perf_counter(); nc(imgs[name]); sync(dev)   # warm-up
        print(f"{name} [{dev}] warm-up run: {time.perf_counter()-t0:.1f}s", flush=True)
        ts = []
        for _ in range(REPEAT):
            t0 = time.perf_counter(); res = nc(imgs[name]); sync(dev); ts.append(time.perf_counter() - t0)
        row[f"torch_{dev}_s"] = min(ts)
        results[(name, dev)] = res
        t0 = time.perf_counter()
        run_fits(paths[name], f"{OUT}/{name}_{dev}.fits", hdu=hdus[name], device=dev, ccl=CCL); sync(dev)
        row[f"torch_{dev}_e2e_s"] = time.perf_counter() - t0
    timing.append(row)
    print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in row.items()}, flush=True)

## 4. Performance

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

T = pd.DataFrame(timing).set_index("image")
T["speedup_cpu_vs_gnu"] = T.gnuastro_s / T.torch_cpu_s
if "torch_cuda_s" in T:
    T["speedup_gpu_vs_gnu"] = T.gnuastro_s / T.torch_cuda_s
    T["speedup_gpu_vs_torchcpu"] = T.torch_cpu_s / T.torch_cuda_s
    T["speedup_gpu_e2e_vs_gnu"] = T.gnuastro_s / T.torch_cuda_e2e_s
display(T.round(3))

cols = [c for c in ["gnuastro_s", "torch_cpu_s", "torch_cuda_s"] if c in T]
labels = {"gnuastro_s": f"Gnuastro ({NTHREADS} threads)", "torch_cpu_s": "torch CPU", "torch_cuda_s": "torch GPU"}
ax = T[cols].rename(columns=labels).plot.bar(figsize=(8, 4), rot=0)
ax.set_ylabel("time per image [s] (lower is better)"); ax.set_title("NoiseChisel run time")
for c in ax.containers: ax.bar_label(c, fmt="%.2f", fontsize=8)
plt.tight_layout(); plt.show()

## 5. Product-by-product comparison

For each frame and each torch device, against Gnuastro's output HDUs `DETECTIONS`, `SKY`, `SKY_STD`, `INPUT-NO-SKY`, plus header values `DETSN` (S/N threshold) and `MEDSTD`.
Sky differences are expressed in units of the local noise (σ = Gnuastro Sky STD); STD differences are relative.

In [ ]:
def load_gnu(path):
    with fits.open(path) as hl:
        d = {"det": hl["DETECTIONS"].data.astype(bool),
             "sky": hl["SKY"].data.astype(np.float32),
             "std": hl["SKY_STD"].data.astype(np.float32),
             "nosky": hl["INPUT-NO-SKY"].data.astype(np.float32),
             "detsn": hl["DETECTIONS"].header.get("DETSN", np.nan),
             "medstd": hl["SKY_STD"].header.get("MEDSTD", np.nan)}
    return d

def torch_products(res, img):
    sky = res.sky_image().cpu().numpy(); std = res.std_image().cpu().numpy()
    return {"det": res.detections.cpu().numpy(), "sky": sky, "std": std,
            "nosky": (img - sky).astype(np.float32),
            "detsn": res.det_sn_threshold, "medstd": res.med_std}

def compare(g, t):
    good = np.isfinite(g["sky"]) & np.isfinite(t["sky"])
    inter = np.count_nonzero(g["det"] & t["det"]); union = np.count_nonzero(g["det"] | t["det"])
    dsky = (t["sky"] - g["sky"])[good] / g["std"][good]
    dstd = (t["std"] - g["std"])[good] / g["std"][good]
    dns = (t["nosky"] - g["nosky"])[good] / g["std"][good]
    return {"mask px differing": int(np.count_nonzero(g["det"] != t["det"])),
            "mask IoU": inter / union if union else 1.0,
            "det frac gnu": g["det"].mean(), "det frac torch": t["det"].mean(),
            "Sky px differing": int(np.count_nonzero(g["sky"][good] != t["sky"][good])),
            "Sky max|diff| [sigma]": float(np.abs(dsky).max()),
            "STD px differing": int(np.count_nonzero(g["std"][good] != t["std"][good])),
            "STD max|rel diff|": float(np.abs(dstd).max()),
            "INPUT-NO-SKY max|diff| [sigma]": float(np.abs(dns).max()),
            "DETSN gnu": g["detsn"], "DETSN torch": t["detsn"], "DETSN diff": t["detsn"] - g["detsn"],
            "MEDSTD rel diff": (t["medstd"] - g["medstd"]) / g["medstd"]}

G = {n: load_gnu(results[(n, "gnuastro")]) for n in paths}
P = {(n, d): torch_products(results[(n, d)], imgs[n]) for n in paths for d in devices}
rows = {f"{n} / torch-{d}": compare(G[n], P[(n, d)]) for n in paths for d in devices}
display(pd.DataFrame(rows).T)

In [ ]:
# GPU vs torch-CPU too: is the CUDA result identical to the CPU one?
if "cuda" in devices:
    rows = {f"{n}: cuda vs cpu": compare(P[(n, "cpu")], P[(n, "cuda")]) for n in paths}
    display(pd.DataFrame(rows).T)

## 6. Difference maps

In [ ]:
dev = devices[-1]   # GPU if available
def binned(a, f=8):  # downsample for display
    H, W = (a.shape[0] // f) * f, (a.shape[1] // f) * f
    return a[:H, :W].reshape(H // f, f, W // f, f).mean(axis=(1, 3))

fig, axs = plt.subplots(len(paths), 4, figsize=(18, 4.3 * len(paths)), squeeze=False)
for i, n in enumerate(paths):
    g, t = G[n], P[(n, dev)]
    ax = axs[i]
    v = np.nanpercentile(g["nosky"], [1, 99.5])
    ax[0].imshow(np.arcsinh(g["nosky"]), cmap="gray", origin="lower",
                 vmin=np.arcsinh(v[0]), vmax=np.arcsinh(v[1])); ax[0].set_title(f"{n}: Gnuastro input-no-sky")
    ax[1].imshow(g["det"].astype(int) - t["det"].astype(int), cmap="bwr", vmin=-1, vmax=1, origin="lower", interpolation="nearest")
    ax[1].set_title(f"mask: Gnuastro - torch-{dev}  ({int((g['det'] != t['det']).sum())} px)")
    ds = (t["sky"] - g["sky"]) / g["std"]; m = max(np.nanmax(np.abs(ds)), 1e-12)
    im = ax[2].imshow(ds, cmap="RdBu_r", vmin=-m, vmax=m, origin="lower"); plt.colorbar(im, ax=ax[2], fraction=0.046)
    ax[2].set_title(f"Sky diff [sigma] (max {m:.1e})")
    dr = (t["std"] - g["std"]) / g["std"]; m = max(np.nanmax(np.abs(dr)), 1e-12)
    im = ax[3].imshow(dr, cmap="RdBu_r", vmin=-m, vmax=m, origin="lower"); plt.colorbar(im, ax=ax[3], fraction=0.046)
    ax[3].set_title(f"Sky STD rel. diff (max {m:.1e})")
    for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
# Per-image histograms of the pixel differences (log counts); a spike at 0 means identical
fig, axs = plt.subplots(len(paths), 3, figsize=(15, 3.3 * len(paths)), squeeze=False)
for i, n in enumerate(paths):
    g, t = G[n], P[(n, dev)]
    for ax, (lab, d) in zip(axs[i], [("Sky diff [sigma]", (t["sky"] - g["sky"]) / g["std"]),
                                     ("STD rel. diff", (t["std"] - g["std"]) / g["std"]),
                                     ("INPUT-NO-SKY diff [sigma]", (t["nosky"] - g["nosky"]) / g["std"])]):
        d = d[np.isfinite(d)]
        ax.hist(d, bins=101, log=True); ax.set_title(f"{n}: {lab}"); ax.set_xlabel(f"max |d| = {np.abs(d).max():.2e}")
plt.tight_layout(); plt.show()

## 7. Summary

In [ ]:
for n in paths:
    r = rows[f"{n} / torch-{devices[-1]}"]
    s = f"{n}: Gnuastro {T.loc[n,'gnuastro_s']:.2f}s | torch CPU {T.loc[n,'torch_cpu_s']:.2f}s"
    if "torch_cuda_s" in T:
        s += f" | torch GPU {T.loc[n,'torch_cuda_s']:.2f}s  ({T.loc[n,'speedup_gpu_vs_gnu']:.1f}x vs Gnuastro, {T.loc[n,'speedup_gpu_vs_torchcpu']:.1f}x vs torch CPU)"
    print(s)
    print(f"    mask px differing: {r['mask px differing']}, Sky px differing: {r['Sky px differing']}, STD px differing: {r['STD px differing']}, DETSN diff: {r['DETSN diff']:+.2e}")